# Hanoi-MM25:  Feature Engineering Pipeline

This notebook reproduces the complete feature engineering pipeline for the **Hanoi-MM25** multimodal PM2.5 forecasting benchmark.

## Feature Categories Created (Total 117 Features):
1. **Temporal Memory & Lags**: Single-day shifts (`shift(1..14)`), rolling means (`rolling(3..14)`), EWM (`ewm(span=3..14)`), temporal differences (`pm25_diff1`, `pm25_diff7`).
2. **K-NN Spatial Neighborhood Lags**: Euclidean spatial distance weighted K-NN daily spatial averages (`pm25_spatial_mean`, `pm25_spatial_lag1..2`).
3. **Physics-Informed Atmospheric Descriptors**: 
   - **Atmospheric Stagnation Index (ASI)**: `1.0 / ((wind_speed + 1.0) * (precip + 1.0))`
   - **Ventilation Coefficient (VC)**: `wind_speed * boundary_layer_height`
   - **Hygroscopic Growth Interaction**: `relative_humidity * AOD`
4. **Pollution Episode Length**: `is_polluted = (pm25 > 50)` lagged by 1 day (`episode_length_lag1`).
5. **Satellite Anomalies & Rolling Lags**: Satellite daily regional anomalies and 7-day rolling means shifted by 1 day (`shift(1)`).
6. **Cyclical Calendar Features**: Day of year sine/cosine encodings, day of week, weekend indicator.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

print("Loading raw processed daily merged dataset...")
df = pd.read_csv("../../data/processed/01_daily_merged_advanced_v3.csv")
df['date'] = pd.to_datetime(df['date'])

if 'latitude_x' in df.columns:
    df = df.rename(columns={'latitude_x': 'latitude', 'longitude_x': 'longitude'})

rename_dict = {
    'wind_speed_10m_kmh_mean': 'wind_speed_mean_kmh',
    'boundary_layer_height_m_mean': 'blh_mean_m',
    'boundary_layer_height_m_max': 'blh_max_m',
    'boundary_layer_height_m_min': 'blh_min_m',
}
df = df.rename(columns=rename_dict)

# Fix missing station coordinates from metadata reference
stations_ref = pd.read_excel('../../data/raw/DataAOD/Hanoi/Stations.xlsx')[['Location', 'Lat', 'Lon']].rename(
    columns={'Location': 'location_id', 'Lat': 'ref_lat', 'Lon': 'ref_lon'})
df['location_id'] = df['location_id'].astype(str)
stations_ref['location_id'] = stations_ref['location_id'].astype(str)
df = df.merge(stations_ref, on='location_id', how='left')
df['latitude'] = df['latitude'].fillna(df['ref_lat'])
df['longitude'] = df['longitude'].fillna(df['ref_lon'])
df = df.drop(columns=['ref_lat', 'ref_lon'])

df = df.sort_values(["location_id", "date"]).reset_index(drop=True)
df["pm25_raw"] = df["pm25"].copy()

# 1. Temporal Memory & Lags (100% Causal)
for lag in [1, 2, 3, 4, 5, 7, 10, 14]:
    df[f"pm25_lag{lag}"] = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(lag))

df["pm25_roll3_mean"] = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df["pm25_roll3_max"]  = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).max())
df["pm25_roll3_min"]  = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).min())
df["pm25_roll3_std"]  = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).std()).fillna(0)

df["pm25_roll7_mean"] = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
df["pm25_roll7_max"]  = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(7, min_periods=1).max())
df["pm25_roll7_median"] = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(7, min_periods=1).median())
df["pm25_roll7_std"]  = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(7, min_periods=1).std()).fillna(0)

df["pm25_roll14_mean"] = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(14, min_periods=1).mean())
df["pm25_roll14_max"]  = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).rolling(14, min_periods=1).max())

df["pm25_ewm3"]  = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).ewm(span=3, adjust=False).mean())
df["pm25_ewm7"]  = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).ewm(span=7, adjust=False).mean())
df["pm25_ewm14"] = df.groupby("location_id")["pm25"].transform(lambda s: s.shift(1).ewm(span=14, adjust=False).mean())

df["pm25_diff1"] = df["pm25_lag1"] - df["pm25_lag2"]
df["pm25_diff7"] = df["pm25_lag1"] - df["pm25_lag7"]
df["pm25_ratio3_7"] = df["pm25_roll3_mean"] / (df["pm25_roll7_mean"] + 1.0)

# 2. Pollution Episode Length (QCVN 05:2023/BTNMT 50 ug/m3 threshold, shifted by 1)
df["is_polluted"] = (df["pm25"] > 50).astype(int)
df["polluted_group"] = df.groupby("location_id")["is_polluted"].transform(lambda s: (s == 0).cumsum())
df["episode_length"] = df.groupby(["location_id", "polluted_group"])["is_polluted"].cumsum()
df["episode_length_lag1"] = df.groupby("location_id")["episode_length"].transform(lambda s: s.shift(1)).fillna(0)

# 3. Physics-Informed Atmospheric Descriptors
precip = df["precipitation_mm"] if "precipitation_mm" in df.columns else df["precip_daily_mm"]
ws = df["wind_speed_mean_kmh"]

df["stagnation_index"] = 1.0 / ((ws + 1.0) * (precip + 1.0))
df["ventilation_coeff"] = ws * df["blh_mean_m"] if "blh_mean_m" in df.columns else 0.0

if "wind_direction_deg" in df.columns:
    rad = np.radians(df["wind_direction_deg"].fillna(0))
    df["wind_u"] = -ws * np.sin(rad)
    df["wind_v"] = -ws * np.cos(rad)

if "relative_humidity_pct_mean" in df.columns and "aod_550_mean" in df.columns:
    df["rh_aod_interaction"] = df["relative_humidity_pct_mean"] * df["aod_550_mean"].fillna(0)

if "blh_mean_m" in df.columns and "relative_humidity_pct_mean" in df.columns:
    df["blh_rh_ratio"] = df["blh_mean_m"] / (df["relative_humidity_pct_mean"] + 1.0)

# 4. Cyclical Calendar Features
df["doy"] = df["date"].dt.dayofyear
df["sin_doy"] = np.sin(2 * np.pi * df["doy"] / 365.25)
df["cos_doy"] = np.cos(2 * np.pi * df["doy"] / 365.25)
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# 5. K-NN Spatial Lags
stations = df[['location_id', 'latitude', 'longitude']].drop_duplicates().set_index('location_id')
nbrs = NearestNeighbors(n_neighbors=min(4, len(stations)), metric='euclidean').fit(stations[['latitude', 'longitude']])
distances, indices = nbrs.kneighbors(stations[['latitude', 'longitude']])
station_iloc_to_id = {i: idx for i, idx in enumerate(stations.index)}
neighbors_dict = {station_iloc_to_id[i]: [station_iloc_to_id[j] for j in row[1:]] for i, row in enumerate(indices)}

pm25_pivot = df.pivot_table(index='date', columns='location_id', values='pm25')
knn_means = pd.Series(index=df.index, dtype=float)
for loc_id in df['location_id'].unique():
    neighbor_ids = neighbors_dict.get(loc_id, [])
    valid_neighbors = [n for n in neighbor_ids if n in pm25_pivot.columns]
    if valid_neighbors:
        loc_knn_daily = pm25_pivot[valid_neighbors].mean(axis=1)
        mask = df['location_id'] == loc_id
        knn_means.loc[mask] = df.loc[mask, 'date'].map(loc_knn_daily)

daily_spatial = df.groupby("date")["pm25"].transform("mean")
df["pm25_spatial_mean"] = knn_means.fillna(daily_spatial)
df["pm25_spatial_lag1"] = df.groupby("location_id")["pm25_spatial_mean"].transform(lambda s: s.shift(1))
df["pm25_spatial_lag2"] = df.groupby("location_id")["pm25_spatial_mean"].transform(lambda s: s.shift(2))

# 6. Satellite Anomalies & 7-day Rolling Features
SAT_COLS_3D = ['co_mean', 'hcho_mean', 'ndvi_buffer_2km', 'ndbi_mean', 'o3_column_density', 'aerosol_index', 'lst_day_c', 'frp_sum_10km', 'nighttime_lights']
AOD_COLS = ['aod_550_mean', 'aod_mean', 'hcho_mean']

for loc_id, grp in df.groupby("location_id"):
    for c in SAT_COLS_3D:
        if c in grp.columns:
            df.loc[grp.index, f"{c}_roll7"] = grp[c].shift(1).rolling(window=7, min_periods=1).mean()
    for c in AOD_COLS:
        if c in grp.columns:
            df.loc[grp.index, f"{c}_roll7"] = grp[c].shift(1).rolling(window=7, min_periods=1).mean()

for c in SAT_COLS_3D:
    if c in df.columns:
        daily_mean = df.groupby("date")[c].transform("mean")
        df[c] = df[c].fillna(daily_mean)
        df[c] = df.groupby("location_id")[c].transform(lambda x: x.ffill())
        df[c] = df.groupby("location_id")[c].transform(lambda x: x.bfill())

for c in SAT_COLS_3D + AOD_COLS:
    if c in df.columns:
        daily_mean = df.groupby("date")[c].transform("mean")
        df[f"{c}_anomaly"] = df[c] - daily_mean
        df[f"{c}_lag1"] = df.groupby("location_id")[c].transform(lambda s: s.shift(1))

print("Feature engineering complete!")
print("Final DataFrame Shape:", df.shape)
